# Tandem-repeat pathogenicity visualizer

**Background, for readers new to this project:** tandem repeats (also called short tandem
repeats, STRs, or microsatellites) are short DNA motifs — here, 3 to 24 base pairs — repeated
back-to-back at a specific place in the genome. Most people carry a fairly stable,
locus-specific number of repeat copies; at a defined set of loci, an abnormally large
expansion of the repeat causes disease. [STRchive](https://strchive.org) is a database hosted by the Dashnow lab that
lists these disease-associated loci, each with a literature-derived **pathogenic threshold**
(the repeat-copy-number cutoff above which the locus is considered expanded), an **evidence
tier** (how well-supported the locus-disease link is: Definitive > Strong > Moderate > Limited
> Disputed), and an inheritance pattern (AD/AR/XD/XR = autosomal/X-linked dominant/recessive).
[TRExplorer v2.0](https://github.com/broadinstitute/trexplorer-catalog) is a separate,
genome-wide catalog — built from several published population-sequencing cohorts — of
repeat-length statistics (population mean, standard deviation, maximum observed, etc.) at
millions of tandem-repeat loci. The three
TRExplorer cohorts used in this notebook are:

- **HPRC256** — long-read genome assemblies from 256 individuals (512 haplotypes), from the
  Human Pangenome Reference Consortium.
- **TenK10K** — a larger, short-read population-sequencing cohort. Short read is more likely to fail to report correct lengths. 
- **AoU1027** — long-read genomes from the NIH *All of Us* Research Program.


**What this notebook asks:** for a known disease locus, how does the *maximum* repeat length
actually observed in an ordinary population sample compare to the
literature pathogenic threshold — and does the locus vary a little or a lot across individuals
in that population (its standard deviation, "population SD")? It then plots those two quantities
against each other.

**Data sources** (all bundled in `data/`, loaded fresh below — see README for full provenance):

- `data/TR_catalog_63loci.json` — the disease catalog this project is built around (gene,
  motif, reference repeat count, region annotation; matched to STRchive for thresholds).
- `data/STRchive_loci.json` — STRchive itself, the source of pathogenic/benign/intermediate
  thresholds, evidence tier, and inheritance pattern. Only loci with evidence tier
  **Moderate or better** (Definitive, Strong, Moderate) are kept — see below.
- `data/TRExplorer_disease_loci_full.json` — the same disease loci, pulled fresh from the
  **full** TRExplorer v2.0 genome-wide catalog (rather than the original 63-locus export), so
  every cohort's max/99th-percentile/n fields are complete.
- `data/TRExplorer_matched_sample.json` — a region-proportionate **background** sample from
  the genome-wide catalog (same CDS/UTR/intron/exon mix as the disease catalog), **not**
  disease-associated. It excludes anything tagged as a known disease locus, is restricted to
  gene-body regions, and restricted to the same motif lengths the disease catalog uses, so the
  comparison is short-tandem-repeat-to-short-tandem-repeat throughout.

This notebook has **two plots**:

1. **Main visualizer** (section 3) — one population cohort at a time (dropdown); marker shape
   encodes genomic region with color showing its broader catagotry. Also provides a 63 loci, region matched random dataset to show how a similar
   propotion compare. These are shown in grey. 
3. **Cross-population comparison** (section 4) — check any combination of HPRC256 / TenK10K /
   AoU1027 to plot them side by side **on the same axes**; marker color *and* shape both
   encode which cohort a point came from, so you can see directly how tight/spread and how far
   from normal the same locus looks across cohorts. Look at some outliers in HPRC256 and see how the 99th percentile compares.

Both plots share the same interactions:
- **Background**: on/off. Disease loci are always shown.
- **Zoom**: drag a box to zoom into a crowded region, scroll to zoom continuously,
  double-click to reset. The toolbar in the plot's top-right corner has a dedicated
  "reset axes" button too.
- **Click a point** to pin its full details (gene, disease, region, population SD,
  fold-change, pathogenic threshold, evidence, inheritance) in the fixed box under that plot
  — the box's left edge is tinted to match the color of the point you clicked, so it's easy to
  tell at a glance which point it's reporting on. It stays there until you click another point
  in the same plot. Hovering just shows the gene name near the cursor, to keep the view
  uncluttered.

Every point — disease and background alike — is plotted as **fold-change**: the maximum
repeat length actually observed in the selected cohort, divided by the reference repeat count
(the repeat-copy number at that locus in the GRCh38 human reference genome assembly). A
fold-change of 1 means "matches the reference exactly"; a fold-change of 5 means "the longest
copy seen in this cohort has 5 times as many repeat units as the reference." Fold-change is
the only basis on which disease and background loci can share the same axis, since background
loci have no pathogenic threshold to compare against.

Run all cells, then use the controls under each plot.

**Setup:** `pip install -r requirements.txt` (see README). Each plot is rendered as a small
plain HTML/JS snippet (not an ipywidgets `FigureWidget`) that loads the plotly.js charting
library from a CDN once and draws into a fixed `<div>`, with a little injected JavaScript
wiring up the click-to-detail panel. This avoids a version-compatibility break some
`FigureWidget` + `ipywidgets` + Jupyter combinations hit, and needs an internet connection the
first time a plot draws (to fetch plotly.js from `cdn.plot.ly`) — after that it's cached by
your browser. If clicking a point doesn't update the details box in your setup, classic
Jupyter Notebook or JupyterLab are the most tested targets.


In [31]:
import json, re, math
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display, HTML, Javascript

DATA_DIR = Path("data")


## 1. Build the per-locus table

This section builds one combined table covering both datasets (disease loci and the
background sample). Each row carries, for each of the three cohorts (HPRC256, TenK10K,
AoU1027): the population standard deviation of repeat count at that locus, the sample size
(`n`, i.e. how many chromosomes/haplotypes were successfully genotyped there), the raw maximum
repeat count observed, and the 99th-percentile repeat count. Fold-change (the metric both
plots use) is derived from the raw maximum; the 99th percentile is computed and kept in the
table too, even though it isn't currently plotted, so the comparison basis is a one-line swap
away from something less sensitive to a single outlier genotype — see the markdown before the
main-plot cell for a concrete example of exactly that happening.

Disease loci are filtered to STRchive evidence tier **Moderate or better** (Definitive,
Strong, Moderate) at this stage, in the table-building step itself — Limited and Disputed
loci are dropped before anything else runs, not just hidden by a filter widget later on.


In [32]:
# The three population cohorts from TRExplorer v2.0 used throughout this notebook. See the
# title cell above for what each one is; T2T assemblies and the sparse Illumina174k cohort are
# deliberately left out.
COHORT_KEYS = ["HPRC256", "TenK10K", "AoU1027"]

# Gencode's region annotation vocabulary for a transcript feature a repeat locus falls in.
REGION_CATEGORIES = ["CDS", "5\' UTR", "3\' UTR", "intron", "exon"]

# BEAN1 (associated with spinocerebellar ataxia 31) is a case where Gencode's gene-region
# annotation resolves the locus to its Ensembl gene ID rather than the gene symbol, while
# TRExplorer's own disease-locus name field uses the symbol -- this remaps the one ID we've
# seen that affects so the two data sources can be joined on gene name.
SPECIAL_GENE_NAME_MAP = {"ENSG00000260851": "BEAN1"}

# Hardcoded floor on STRchive's evidence tier for a locus-disease association (Definitive >
# Strong > Moderate > Limited > Disputed). There is no selector widget for this -- Limited and
# Disputed loci never enter the table in the first place.
EVIDENCE_KEEP = {"Definitive", "Strong", "Moderate"}


def norm_chrom(c):
    """Strip the 'chr' prefix some sources use, so chromosome names compare equal regardless
    of which convention ('chr7' vs '7') a given file happens to use."""
    return c.replace("chr", "")


def parse_region(reference_region):
    """Parse a 'chr7:1234-5678'-style coordinate string into (chromosome, start, end)."""
    chrom, rest = reference_region.split(":")
    start, end = rest.split("-")
    return norm_chrom(chrom), int(start), int(end)


def to_float(v):
    if v is None or v == "":
        return None
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


def parse_hist_pairs(histstr):
    """TRExplorer encodes a cohort's genotype distribution at a locus as a compact histogram
    string, e.g. '12x:1,14x:3,...' meaning '1 allele was genotyped at 12 repeat units, 3
    alleles at 14 repeat units, ...'. This parses that string into a sorted list of
    (repeat_count, allele_count) pairs, which the two helpers below turn into summary stats."""
    if not histstr:
        return []
    pairs = []
    for tok in histstr.split(","):
        tok = tok.strip()
        m = re.match(r"(-?\d+)x:(\d+)", tok)
        if m:
            pairs.append((int(m.group(1)), int(m.group(2))))
    return sorted(pairs)


def hist_sd_n(pairs):
    """Population standard deviation and total allele count (n) of repeat copy number,
    computed directly from a genotype histogram (no raw per-individual list is available for
    every cohort, just this binned count)."""
    if not pairs:
        return None, None
    n = sum(c for _, c in pairs)
    mean = sum(v * c for v, c in pairs) / n
    var = sum(c * (v - mean) ** 2 for v, c in pairs) / n
    return math.sqrt(var), n


def hist_percentile(pairs, q):
    """The repeat-count value at the qth percentile (e.g. q=99) of a genotype histogram."""
    if not pairs:
        return None
    total = sum(c for _, c in pairs)
    rank = max(1, math.ceil(q / 100.0 * total))
    cum = 0
    for val, cnt in pairs:
        cum += cnt
        if cum >= rank:
            return val
    return pairs[-1][0]


def extract_cohort_stats(rec):
    """Pull population SD, sample size, max observed repeat count, and 99th percentile for
    each of the three cohorts out of one TRExplorer record. Works the same way whether the
    record is a disease locus (data/TRExplorer_disease_loci_full.json) or a background-sample
    locus (data/TRExplorer_matched_sample.json) -- both were pulled from the same genome-wide
    catalog and share the same field names. Different cohorts ship different raw material:
    HPRC256/TenK10K carry a full genotype histogram per locus, so their stats are computed
    directly from that; AoU1027 ships only pre-computed summary statistics (no raw histogram,
    for participant-privacy reasons on the data provider's end), so its stats are read
    straight off those fields instead."""
    stats_by_cohort = {}

    for cohort in ["HPRC256", "TenK10K"]:
        hist = rec.get(f"{cohort}_AlleleHistogram")
        pairs = parse_hist_pairs(hist) if hist else []
        sd, n = hist_sd_n(pairs)
        if sd is None:
            sd = to_float(rec.get(f"{cohort}_Stdev"))
        if n is None:
            n = rec.get(f"{cohort}_NumCalledAlleles")
        max_allele = rec.get(f"{cohort}_MaxAllele")
        p99 = to_float(rec.get(f"{cohort}_99thPercentile"))
        if pairs and p99 is None:
            p99 = hist_percentile(pairs, 99)
        if sd is not None or max_allele is not None:
            stats_by_cohort[cohort] = {"n": n, "sd": sd, "max": max_allele, "p99": p99}

    aou_sd = to_float(rec.get("AoU1027_Stdev"))
    aou_max = rec.get("AoU1027_MaxAllele")
    if aou_sd is not None or aou_max is not None:
        stats_by_cohort["AoU1027"] = {
            "n": rec.get("AoU1027_NumCalledAlleles"), "sd": aou_sd,
            "max": aou_max, "p99": to_float(rec.get("AoU1027_99thPercentile")),
        }

    return stats_by_cohort


def classify_region(gencode_region):
    """Collapse Gencode's fine-grained region annotation into three coarse buckets: coding
    (CDS, i.e. protein-coding sequence), noncoding (5'/3' UTR or intron), or ambiguous (a
    bare 'exon' annotation, meaning this position is coding in some transcript isoforms of the
    gene and noncoding in others, depending on splicing)."""
    if gencode_region == "CDS":
        return "coding"
    if gencode_region == "exon":
        return "ambiguous"
    return "noncoding"


def find_strchive_match(locus, strchive):
    """Match one row of the disease catalog to its STRchive entry by chromosome + gene symbol
    first; if that fails, fall back to the STRchive entry on the same chromosome whose
    coordinates are closest, since a small number of loci have inconsistent gene-symbol
    annotations between the two sources."""
    chrom, start0, end1 = parse_region(locus["ReferenceRegion"])
    gene = locus["GencodeGeneName"]
    candidates = [s for s in strchive if norm_chrom(s["chrom"]) == chrom and s["gene"] == gene]
    if not candidates:
        candidates = [s for s in strchive if norm_chrom(s["chrom"]) == chrom]
    best, best_score = None, float("inf")
    for s in candidates:
        score = abs(s["start_hg38"] - start0) + abs(s["stop_hg38"] - end1)
        if score < best_score:
            best, best_score = s, score
    return best


def find_disease_full_match(locus, disease_full):
    """Match one row of the disease catalog to its record in the full TRExplorer genome-wide
    disease-locus export. TRExplorer's own disease-locus name field
    (KnownDiseaseAssociatedLocus) is usually a plain gene symbol, with two exceptions this
    data has: BEAN1 (see SPECIAL_GENE_NAME_MAP above) and genes with more than one
    disease-relevant locus (ARX, HOXA13 each have entries suffixed _1/_2/_3), which need
    coordinate disambiguation rather than a straight name match."""
    gene = locus["GencodeGeneName"]
    target = SPECIAL_GENE_NAME_MAP.get(gene, gene).upper()
    chrom0, start0, end0 = parse_region(locus["ReferenceRegion"])

    candidates = []
    for r in disease_full:
        kdal = (r.get("KnownDiseaseAssociatedLocus") or "").upper()
        if kdal == target or kdal.split("_")[0] == target:
            candidates.append(r)
    if not candidates:
        return None
    if len(candidates) == 1:
        return candidates[0]
    best, best_score = None, float("inf")
    for r in candidates:
        c, s, e = parse_region(r["ReferenceRegion"])
        if c != chrom0:
            continue
        score = abs(s - start0) + abs(e - end0)
        if score < best_score:
            best, best_score = r, score
    return best


def build_disease_df(catalog_path=DATA_DIR / "TR_catalog_63loci.json",
                      strchive_path=DATA_DIR / "STRchive_loci.json",
                      disease_full_path=DATA_DIR / "TRExplorer_disease_loci_full.json"):
    """Build one row per disease locus: STRchive's thresholds/evidence/inheritance, joined to
    the locus's per-cohort population statistics from the full TRExplorer export. Loci below
    the EVIDENCE_KEEP floor are dropped here, before the row is ever created."""
    catalog = json.load(open(catalog_path))
    strchive = json.load(open(strchive_path))
    disease_full = json.load(open(disease_full_path))

    rows = []
    for locus in catalog:
        strc = find_strchive_match(locus, strchive)
        if strc is None:
            continue
        evidence = (strc.get("evidence") or [None])[0]
        if evidence not in EVIDENCE_KEEP:
            continue
        dfull = find_disease_full_match(locus, disease_full)
        cohort_stats = extract_cohort_stats(dfull) if dfull is not None else {}

        rows.append({
            "dataset": "disease",
            "gene": locus["GencodeGeneName"],
            "disease": strc["disease"],
            "disease_id": strc["id"],
            "locus_id": locus["LocusId"],
            "motif": locus["ReferenceMotif"],
            "motif_len_bp": int(locus["MotifSize"]),
            "ref_repeats": int(locus["NumRepeatsInReference"]),
            "region_catalog": locus.get("GencodeGeneRegion"),
            "region_class": classify_region(locus.get("GencodeGeneRegion")),
            "region_strchive": strc.get("location_in_gene"),
            "inheritance": strc.get("inheritance") or [],
            "evidence": evidence,
            "benign_max": strc.get("benign_max"),
            "intermediate_min": strc.get("intermediate_min"),
            "intermediate_max": strc.get("intermediate_max"),
            "pathogenic_min": strc.get("pathogenic_min"),
            "pathogenic_max": strc.get("pathogenic_max"),
            "cohort_stats": cohort_stats,
        })
    return pd.DataFrame(rows)


def build_background_df(sample_path, dataset_label):
    """Build one row per background-sample locus (non-disease, region-proportionate to the
    disease catalog -- see the title cell). These loci have no STRchive entry by definition,
    so their disease/threshold/evidence/inheritance fields are all left blank; only their
    TRExplorer population statistics are populated."""
    records = json.load(open(sample_path))
    rows = []
    for rec in records:
        cohort_stats = extract_cohort_stats(rec)
        rows.append({
            "dataset": dataset_label,
            "gene": rec.get("GencodeGeneName"),
            "disease": None,
            "disease_id": None,
            "locus_id": rec.get("LocusId"),
            "motif": rec.get("Motif"),
            "motif_len_bp": rec.get("MotifSize"),
            "ref_repeats": rec.get("NumRepeatsInReference"),
            "region_catalog": rec.get("GencodeGeneRegion"),
            "region_class": classify_region(rec.get("GencodeGeneRegion")),
            "region_strchive": rec.get("GencodeGeneRegion"),
            "inheritance": [],
            "evidence": None,
            "benign_max": None,
            "intermediate_min": None,
            "intermediate_max": None,
            "pathogenic_min": None,
            "pathogenic_max": None,
            "cohort_stats": cohort_stats,
        })
    return pd.DataFrame(rows)


disease_df = build_disease_df()
matched_df = build_background_df(DATA_DIR / "TRExplorer_matched_sample.json", "matched_background")
df = pd.concat([disease_df, matched_df], ignore_index=True)

print(f"disease (Moderate+ evidence only): {len(disease_df)} loci, "
      f"background (proportionate): {len(matched_df)} loci")
for cohort in COHORT_KEYS:
    for name, sub in [("disease", disease_df), ("background", matched_df)]:
        n_have = sub["cohort_stats"].apply(lambda d: cohort in d).sum()
        print(f"  {name:10s} x {cohort:8s}: {n_have}/{len(sub)} loci have data")
df.head()


disease (Moderate+ evidence only): 52 loci, background (proportionate): 63 loci
  disease    x HPRC256 : 52/52 loci have data
  background x HPRC256 : 63/63 loci have data
  disease    x TenK10K : 44/52 loci have data
  background x TenK10K : 63/63 loci have data
  disease    x AoU1027 : 45/52 loci have data
  background x AoU1027 : 63/63 loci have data


C:\Users\nicol\AppData\Local\Temp\ipykernel_22924\3641949839.py:264: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,dataset,gene,disease,disease_id,locus_id,motif,motif_len_bp,ref_repeats,region_catalog,region_class,region_strchive,inheritance,evidence,benign_max,intermediate_min,intermediate_max,pathogenic_min,pathogenic_max,cohort_stats
0,disease,VWA1,"Neuronopathy, distal hereditary motor, autosom...",HMNR7_VWA1,1-1435798-1435818-GGCGCGGAGC,GGCGCGGAGC,10,2,CDS,coding,Coding Exon 1,[AR],Definitive,2.0,NaN,NaN,1,3,"{'HPRC256': {'n': 512, 'sd': 0.044150994357255..."
1,disease,DAB1,Spinocerebellar ataxia type 37,SCA37_DAB1,1-57367043-57367118-AAAAT,AAAAT,5,15,intron,noncoding,Intron 1 (most isoforms),[AD],Strong,30.0,NaN,NaN,31,75,"{'HPRC256': {'n': 512, 'sd': 58.47894951707916..."
2,disease,ABCD3,Oculopharyngodistal myopathy type 5,OPDM5_ABCD3,1-94418421-94418442-GCC,GCC,3,7,5' UTR,noncoding,5' UTR,[AD],Moderate,44.0,NaN,NaN,118,694,"{'HPRC256': {'n': 512, 'sd': 1.085431536806329..."
3,disease,NOTCH2NLC,"Neuronal intranuclear inclusion disease, Alzhe...",NIID_NOTCH2NLC,1-149390802-149390841-GGC,GGC,3,13,CDS,coding,5' UTR,[AD],Definitive,37.0,38.0,65.0,66,517,"{'HPRC256': {'n': 511, 'sd': 4.838035684279659..."
4,disease,STARD7,Familial adult myoclonic epilepsy 2,FAME2_STARD7,2-96197066-96197121-AAAAT,AAAAT,5,11,intron,noncoding,Intron 1,[AD],Strong,NaN,NaN,NaN,274,558,"{'HPRC256': {'n': 512, 'sd': 60.18233400478439..."


## 2. Shared plot rendering helper

Both plots below (the main single-cohort view and the cross-population comparison) draw
through the same small rendering function, so clicking, zooming, and the color-matched detail
box behave identically in both places. It loads the plotly.js charting library from a CDN
exactly once per notebook session (a tiny `<script src=...>` tag, not an inline copy of the
~4MB library), draws the figure into a fixed-id `<div>`, and wires up a `plotly_click`
listener that writes the clicked point's details into a fixed box below the plot and tints
that box's left edge to match the point's color — all in plain JavaScript, with no
`FigureWidget`/ipywidgets Comm channel involved for the plot itself. The JS polls briefly for
`Plotly` to exist before drawing, so it's safe even if the CDN script is still loading the
first time a plot appears.


In [33]:
PLOTLYJS_CDN = "https://cdn.plot.ly/plotly-2.35.2.min.js"
_plotlyjs_tag_injected = {"done": False}


def ensure_plotlyjs_tag():
    """Insert the <script> tag that loads plotly.js, but only the first time any plot draws
    -- repeating it on every redraw would be wasteful and isn't necessary once the browser has
    the library loaded."""
    if not _plotlyjs_tag_injected["done"]:
        display(HTML(f'<script src="{PLOTLYJS_CDN}"></script>'))
        _plotlyjs_tag_injected["done"] = True


def detail_box_html(box_id, placeholder="Click a point above for details."):
    """The fixed-size details panel shown under each plot. It starts with a neutral border;
    render_figure's click handler re-colors the left edge to match whichever point was
    clicked, so the box visually traces back to its source point."""
    return HTML(
        f'''<div id="{box_id}"
             style="border:8px solid #d8d7d2; border-left:8px solid #d8d7d2;
                    border-radius:8px; padding:14px 16px; margin-top:8px;
                    width:820px; max-width:100%; height:160px; overflow-y:auto;
                    font-family:Consolas,\'SFMono-Regular\',monospace;
                    font-size:13px; line-height:1.5; background:#fafaf8; color:#2b2a28;">
             {placeholder}
         </div>'''
    )


def render_figure(fig, div_id, detail_box_id):
    """Draw `fig` into a fixed-id <div> via plain HTML/JS (not display(fig)), so the div id is
    known precisely and a native plotly_click listener can be attached deterministically, in
    the same script block that draws the plot -- no separate "find the newest plot div" step,
    and no FigureWidget. Each point's customdata is expected to carry
    [gene_name, detail_html, point_color]: gene_name drives the minimal hover label, detail_html
    is written into the fixed box on click, and point_color re-tints that box's left edge to
    match."""
    ensure_plotlyjs_tag()
    fig_json = fig.to_json()
    config_json = json.dumps({"scrollZoom": True, "displaylogo": False})
    html_block = f'<div id="{div_id}" style="width:100%;"></div>'
    js_block = f'''
    (function() {{
      function draw() {{
        if (typeof Plotly === "undefined") {{ setTimeout(draw, 80); return; }}
        var el = document.getElementById("{div_id}");
        if (!el) return;
        var figData = {fig_json};
        Plotly.newPlot(el, figData.data, figData.layout, {config_json});
        el.on("plotly_click", function(data) {{
          if (!data || !data.points || !data.points.length) return;
          var pt = data.points[0];
          if (!pt.customdata) return;
          var detail = pt.customdata[1];
          var color = pt.customdata[2];
          var box = document.getElementById("{detail_box_id}");
          if (box && detail) {{
            box.innerHTML = detail;
            if (color) {{ box.style.borderLeftColor = color; box.style.borderColor = color; }}
          }}
        }});
      }}
      draw();
    }})();
    '''
    display(HTML(html_block))
    display(Javascript(js_block))


## 3. Main visualizer

One population cohort at a time. Marker **color** encodes the coarse region class for disease
loci (coding / noncoding / ambiguous — see `classify_region` above for exactly what those
mean), with a flat gray for the background sample. Marker **shape** encodes the finer-grained
Gencode region (CDS, 5\' UTR, 3\' UTR, intron, or an isoform-ambiguous exon). A light
opacity/size step keeps the background sample reading as context rather than signal.

**Why there's no "Pooled" population-source option, and no T2T or Illumina174k:** combining
population SD (or, worse, the maximum observed repeat length) across "whichever cohorts
happen to have data for this locus" would mean a locus covered by more, or larger, cohorts
gets more chances to show an extreme value than one covered by fewer — a sample-size artifact,
not biology. Picking one specific cohort at a time keeps every compared locus on equal
footing. T2T assemblies and the Illumina174k cohort are both dropped to keep the source list
to the three cleanest, most complete cohorts for this comparison. (If you want to see cohorts
side by side instead of one at a time, that's section 4 below.)

**On "fold-change" (the only metric used here):** every point's y-value is the maximum repeat
length actually seen in the *selected single cohort*, divided by the reference repeat count.
This is the only basis on which disease and background loci can share the same y-axis, since
background loci have no pathogenic threshold at all to be measured against. Interpret it as
"how far does this exact locus actually stretch in an ordinary, mostly-unaffected population
sample" — a different question from "how far is the known disease threshold." For disease loci
the two numbers are usually very different: a cohort of a few hundred to ~1,000 people
essentially never contains a symptomatic full-expansion carrier of a rare disease. Click any
disease point to see its literature `pathogenic_min` alongside its max-observed fold-change in
the detail box.

Raw maximum is itself an outlier-sensitive statistic — a single genotyping or assembly
artifact can swing it far more than it would swing the standard deviation. A concrete example
already in this data: DAB1 (a gene with no established disease link at this locus, used here
purely to illustrate the artifact) has, in HPRC256, a mode and median repeat count of 13 but a
99th percentile of 226 and a **maximum of 1,003**, driven by a single outlier haplotype. The
table computes and carries a `p99` (99th percentile) value alongside `max` for every cohort
precisely so the comparison basis could be swapped to something less outlier-prone later,
without re-deriving anything — this notebook currently always uses the raw maximum, per the
analysis plan this project is following.


In [34]:
VWA1_GENE = "VWA1"  # a repeat *contraction*, not an expansion, drives disease at this locus
                     # -- it doesn't fit the "larger fold-change = more concerning" framing
                     # every other locus in this catalog uses, so it's excluded throughout.

COARSE_COLORS = {
    "coding": "#2a78d6",
    "noncoding": "#eb6834",
    "ambiguous": "#1baf7a",
}
COARSE_LABELS = {
    "coding": "Coding (CDS)",
    "noncoding": "Noncoding (UTR / intron)",
    "ambiguous": "Ambiguous / isoform-dependent",
}
# Marker shape for fine-grained Gencode region, used by both plots in this notebook wherever
# region (rather than cohort) is the thing being shape-encoded.
FINE_SYMBOLS = {
    "CDS": "circle",
    "5\' UTR": "triangle-up",
    "3\' UTR": "triangle-down",
    "intron": "square",
    "exon": "diamond",
}
DEFAULT_SYMBOL = "circle"
BACKGROUND_COLOR = "#6b6b66"
DATASET_OPACITY = {"disease": 1.0, "matched_background": 0.8}
DATASET_SIZE = {"disease": 11, "matched_background": 9}
DATASET_LABELS = {"disease": "Disease loci", "matched_background": "Background (proportionate)"}

# VWA1 is excluded from every plot in this notebook (see the comment on VWA1_GENE above).
base_df = df[df["gene"] != VWA1_GENE].copy()


def cohort_completeness_label(cohort):
    """A short 'N/total' summary of how many loci in each dataset actually have data for this
    cohort, shown on the cohort-picker widgets so it's clear up front when a cohort will drop
    some loci (see the AoU1027 caveat at the bottom of the notebook for why that happens)."""
    parts = []
    for dset in ["disease", "matched_background"]:
        sub = base_df[base_df["dataset"] == dset]
        n = int(sub["cohort_stats"].apply(lambda d: cohort in d).sum())
        parts.append(f"{n}/{len(sub)}")
    return f"{cohort}  (disease {parts[0]} · background {parts[1]})"


COHORT_LABELS = {c: cohort_completeness_label(c) for c in COHORT_KEYS}

# ---- widgets -----------------------------------------------------------
show_background_w = widgets.Checkbox(value=False, description="Background (proportionate)")

cohort_source_w = widgets.Dropdown(
    options=[(COHORT_LABELS[c], c) for c in COHORT_KEYS],
    value="HPRC256", description="Population source",
    style={"description_width": "160px"}, layout=widgets.Layout(width="560px"),
)

exclude_vwa1_note = widgets.HTML("<i>VWA1 (a repeat contraction, not an expansion) is always excluded from this view.</i>")

plot_out = widgets.Output()
MAIN_PLOT_DIV_ID = "tr-plot-div-main"
MAIN_DETAIL_BOX_ID = "tr-detail-box-main"


def detail_html(r, cohort_label):
    """The full, human-readable description of one point, written into the fixed details box
    when that point is clicked. One bullet per field, so individual lines are easy to edit,
    reorder, or add to by hand later."""
    dataset_label = DATASET_LABELS[r["dataset"]]
    n_str = f"{int(r['pop_n'])}" if pd.notna(r.get("pop_n")) else "n/a"
    patho = r.get("pathogenic_min")
    patho_str = f"{patho:.0f}" if pd.notna(patho) else "n/a"
    header = (f"<b>{r['gene']}</b> ({dataset_label})"
              + (f" — {r['disease']}" if r.get("disease") else ""))
    items = [
        f"region: {r['region_strchive']} ({r['region_class']}, {r['region_catalog']})",
        f"reference repeats: {r['ref_repeats']}",
        f"max observed: {r['basis_value']:.0f} (99th percentile: {r.get('pop_p99')})",
        f"pathogenic threshold: {patho_str}",
        f"population SD: {r['pop_sd']:.2f} (n={n_str}, {cohort_label})",
        f"fold-change: {r['fold_change']:.2f}",
        f"evidence tier: {r['evidence']}",
        f"inheritance: {','.join(r['inheritance'] or []) or 'n/a'}",
    ]
    bullets = "".join(f"<li>{item}</li>" for item in items)
    return f'{header}<ul style="margin:4px 0 0 0; padding-left:18px;">{bullets}</ul>'


def point_customdata(rows, cohort_label, color):
    """[gene, full_detail_html, point_color] per point -- see render_figure's docstring for
    how each of the three is used (gene for the hover label, detail/color for the click-driven
    details box). `color` is passed in once per call rather than read per row, since every
    point drawn by a single go.Scatter trace already shares one marker color."""
    return [[r["gene"], detail_html(r, cohort_label), color] for _, r in rows.iterrows()]


def make_main_figure(sub, cohort_label):
    fig = go.Figure()

    disease_sub = sub[sub["dataset"] == "disease"]
    for cls in ["noncoding", "coding", "ambiguous"]:
        csub = disease_sub[disease_sub["region_class"] == cls]
        if len(csub) == 0:
            continue
        symbols = csub["region_catalog"].map(FINE_SYMBOLS).fillna(DEFAULT_SYMBOL)
        color = COARSE_COLORS[cls]
        fig.add_trace(go.Scatter(
            x=csub["_x"], y=csub["_y"], mode="markers", name=f"Disease – {COARSE_LABELS[cls]}",
            marker=dict(color=color, size=DATASET_SIZE["disease"], symbol=symbols,
                        opacity=DATASET_OPACITY["disease"], line=dict(color="white", width=1)),
            customdata=point_customdata(csub, cohort_label, color),
            hovertemplate="<b>%{customdata[0]}</b><extra></extra>",
        ))

    bsub = sub[sub["dataset"] == "matched_background"]
    if len(bsub) > 0:
        symbols = bsub["region_catalog"].map(FINE_SYMBOLS).fillna(DEFAULT_SYMBOL)
        fig.add_trace(go.Scatter(
            x=bsub["_x"], y=bsub["_y"], mode="markers", name=DATASET_LABELS["matched_background"],
            marker=dict(color=BACKGROUND_COLOR, size=DATASET_SIZE["matched_background"], symbol=symbols,
                        opacity=DATASET_OPACITY["matched_background"], line=dict(color="white", width=1)),
            customdata=point_customdata(bsub, cohort_label, BACKGROUND_COLOR),
            hovertemplate="<b>%{customdata[0]}</b><extra></extra>",
        ))

    # fold_change = basis_value / ref_repeats is always strictly positive (a ratio of two
    # positive repeat counts), so a log y-axis is safe and is the appropriate scale for a
    # ratio spanning orders of magnitude. dtick=1 means "one labeled tick per decade" --
    # without it, Plotly's automatic minor-tick labels on a wide log range print bare
    # mantissas ("5", "2") at several different decades with no way to tell which apart: a
    # real misread risk on this axis.
    fig.update_layout(
        template="plotly_white",
        height=600,
        dragmode="zoom",
        title=f"Population Spread x Fold Change from Reference — {cohort_label} · max observed",
        xaxis_title="Population SD of repeat copy number (repeat units)",
        yaxis=dict(title="Fold-change (max observed ÷ reference)", type="log", dtick=1),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(t=100, r=40, l=70, b=60),
    )
    return fig


def compute_cohort_rows(datasets_shown, cohort):
    """For one cohort, pull its population SD/n/max/99th-percentile for every locus in
    `datasets_shown`, derive fold-change from the max, and drop any locus missing data for
    this specific cohort (tracked loci that were never genotyped there, or didn't pass the
    fold-change computation because the reference repeat count was zero)."""
    part = base_df[base_df["dataset"].isin(datasets_shown)].copy()
    entry = part["cohort_stats"].apply(lambda d: d.get(cohort))
    part["pop_sd"] = pd.to_numeric(entry.apply(lambda s: s["sd"] if s else None), errors="coerce")
    part["pop_n"] = pd.to_numeric(entry.apply(lambda s: s["n"] if s else None), errors="coerce")
    part["pop_max"] = pd.to_numeric(entry.apply(lambda s: s["max"] if s else None), errors="coerce")
    part["pop_p99"] = pd.to_numeric(entry.apply(lambda s: s["p99"] if s else None), errors="coerce")
    part = part[part["pop_sd"].notna()]

    part["basis_value"] = pd.to_numeric(part["pop_max"], errors="coerce")
    part = part[part["basis_value"].notna()]

    ref_repeats_num = pd.to_numeric(part["ref_repeats"], errors="coerce")
    has_ref = ref_repeats_num > 0
    part["fold_change"] = pd.to_numeric(
        np.where(has_ref, part["basis_value"] / ref_repeats_num, np.nan), errors="coerce")
    part = part[part["fold_change"].notna()]
    part["cohort"] = cohort
    return part.assign(_x=part["pop_sd"], _y=part["fold_change"])


def update(*_):
    cohort = cohort_source_w.value
    datasets_shown = ["disease"]
    if show_background_w.value:
        datasets_shown.append("matched_background")

    sub = compute_cohort_rows(datasets_shown, cohort)
    fig = make_main_figure(sub, cohort)
    plot_out.clear_output(wait=True)
    with plot_out:
        render_figure(fig, MAIN_PLOT_DIV_ID, MAIN_DETAIL_BOX_ID)


for w in [cohort_source_w, show_background_w]:
    w.observe(update, names="value")

update()

controls = widgets.VBox([
    show_background_w,
    cohort_source_w,
    exclude_vwa1_note,
])

display(controls, plot_out)
display(detail_box_html(MAIN_DETAIL_BOX_ID))

import plotly.express as px





Output()

AttributeError: module 'numpy' has no attribute 'bool8'

In [ ]:
import pandas as pd

# Default columns for a slide table (the full df has ~24, too wide to read on a slide).
SLIDE_COLS = ["gene", "disease", "motif", "region_class", "evidence", "pathogenic_min"] + \
             [f"{c} SD" for c in COHORT_KEYS]


def flatten_for_slide(data, cols=None):
    """Turn the notebook's df into plain columns: expands cohort_stats into
    '<cohort> SD' / '<cohort> max' and joins list columns like inheritance."""
    d = data.copy()
    if "cohort_stats" in d.columns:
        for c in COHORT_KEYS:
            d[f"{c} SD"] = d["cohort_stats"].apply(
                lambda s: round(s[c]["sd"], 2) if c in s and s[c].get("sd") is not None else "")
            d[f"{c} max"] = d["cohort_stats"].apply(lambda s: s.get(c, {}).get("max", ""))
        d = d.drop(columns="cohort_stats")
    for col in d.columns:
        if d[col].apply(lambda v: isinstance(v, (list, tuple))).any():
            d[col] = d[col].apply(lambda v: ", ".join(map(str, v)) if isinstance(v, (list, tuple)) else v)
    if cols is None:
        cols = [c for c in SLIDE_COLS if c in d.columns] or list(d.columns)
    elif cols == "all":
        cols = list(d.columns)
    d = d[cols]
    return d.fillna("").astype(str)


def table_figure(data, cols=None, max_rows=15, base_font=14, max_chars=28):
    """Render a DataFrame as a clean slide-ready table figure.
    cols=None uses SLIDE_COLS, cols="all" shows every column, or pass a list."""
    d = flatten_for_slide(data, cols)
    if len(d) > max_rows:
        print(f"Showing first {max_rows} of {len(d)} rows (set max_rows= to change).")
        d = d.head(max_rows)
    d = d.apply(lambda s: s.str.slice(0, max_chars))          # trim long text (e.g. disease names)
    fig, ax = plt.subplots(figsize=(max(8, 1.6 * len(d.columns)), 0.5 * (len(d) + 2)))
    ax.axis("off")
    tbl = ax.table(cellText=d.values, colLabels=list(d.columns), loc="center", cellLoc="left")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(base_font)
    tbl.auto_set_column_width(range(len(d.columns)))
    tbl.scale(1, 1.6)
    for (r, _), cell in tbl.get_celld().items():
        cell.set_edgecolor("#dddddd")
        if r == 0:
            cell.set_text_props(weight="bold", color="white")
            cell.set_facecolor("#33415c")
        elif r % 2 == 0:
            cell.set_facecolor("#f3f4f7")
    return fig


def export_display(obj=None, cols=None, max_rows=15, name="tr_visualizer", **kwargs):
    """Presentation copy of whatever is displayed: a DataFrame (e.g. df.head()),
    a matplotlib/plotly figure, or (default) the last open matplotlib figure."""
    if isinstance(obj, pd.DataFrame):
        fig = table_figure(obj, cols=cols, max_rows=max_rows)
        kwargs.setdefault("clean_spines", False)
        kwargs.setdefault("width_in", fig.get_figwidth())    # keep the table's own shape
        kwargs.setdefault("height_in", fig.get_figheight())
        out = export_presentation(fig, name=name + "_table", **kwargs)
        plt.close(fig)
        return out
    return export_presentation(obj, name=name, **kwargs)

## 4. Cross-population comparison

A second, independent plot: check any combination of the three population sources to see the
same loci plotted at each checked cohort's own population SD and fold-change, side by side
**on the same axes** — so a locus that's tight in HPRC256 but spread out in TenK10K shows up
as two differently-colored, differently-shaped points at different x-positions on one chart,
rather than having to flip the main view's dropdown back and forth.

Marker **color and shape both encode cohort** here (circle = HPRC256, triangle = TenK10K,
square = AoU1027, each in its own color), rather than splitting those two channels between
cohort and region the way the main plot does — the redundancy makes it easier to tell cohorts
apart at a glance, especially once points from different cohorts start overlapping. The
trade-off is that region class and fine-grained region are no longer shown visually at all in
this plot; they're still recorded and shown in the click-detail text, just not as color or
shape here. Disease loci are drawn at full size/opacity and background-sample loci smaller and
more transparent, the same convention the main plot uses to keep background reading as
context.

**Why this is a separate plot rather than replacing the main one:** the main view (section 3)
keeps one cohort at a time and uses marker shape for fine-grained region, which is the more
common everyday use; this view trades that off for direct cohort-to-cohort comparability when
that's specifically what's needed.


In [ ]:
COHORT_ORDER = ["HPRC256", "TenK10K", "AoU1027"]
# Both color and shape encode cohort in this plot (see the markdown above for why); chosen to
# be visually distinct from the region-class colors (blue/orange/green) the main plot uses, so
# the two plots' color meanings don't get confused with each other when viewed close together.
COHORT_COLORS = {"HPRC256": "#7b4fa6", "TenK10K": "#0e7c86", "AoU1027": "#c98a1b"}
COHORT_SYMBOLS = {"HPRC256": "circle", "TenK10K": "triangle-up", "AoU1027": "square"}

cross_plot_out = widgets.Output()
CROSS_PLOT_DIV_ID = "tr-plot-div-cross"
CROSS_DETAIL_BOX_ID = "tr-detail-box-cross"

show_background_cross_w = widgets.Checkbox(value=False, description="Background (proportionate)")

cohort_check_w = {
    c: widgets.Checkbox(value=(c == "HPRC256"), description=COHORT_LABELS[c],
                         layout=widgets.Layout(width="400px"))
    for c in COHORT_KEYS
}


def cross_detail_html(r):
    """Same idea as detail_html above, but for this plot's points -- each of which belongs to
    exactly one cohort (recorded in r['cohort']), since a locus can appear once per checked
    cohort here rather than once overall."""
    dataset_label = DATASET_LABELS[r["dataset"]]
    n_str = f"{int(r['pop_n'])}" if pd.notna(r.get("pop_n")) else "n/a"
    patho = r.get("pathogenic_min")
    patho_str = f"{patho:.0f}" if pd.notna(patho) else "n/a"
    header = (f"<b>{r['gene']}</b> ({dataset_label})"
              + (f" — {r['disease']}" if r.get("disease") else ""))
    items = [
        f"cohort: <b>{r['cohort']}</b>",
        f"region: {r['region_strchive']} ({r['region_class']}, {r['region_catalog']})",
        f"reference repeats: {r['ref_repeats']}",
        f"max observed: {r['basis_value']:.0f} (99th percentile: {r.get('pop_p99')})",
        f"pathogenic threshold: {patho_str}",
        f"population SD: {r['pop_sd']:.2f} (n={n_str})",
        f"fold-change: {r['fold_change']:.2f}",
        f"evidence tier: {r['evidence']}",
        f"inheritance: {','.join(r['inheritance'] or []) or 'n/a'}",
    ]
    bullets = "".join(f"<li>{item}</li>" for item in items)
    return f'{header}<ul style="margin:4px 0 0 0; padding-left:18px;">{bullets}</ul>'


def cross_customdata(rows, color):
    return [[r["gene"], cross_detail_html(r), color] for _, r in rows.iterrows()]


def make_cross_figure(sub, cohorts_shown):
    """One trace per (dataset, cohort) combination that's actually present, each trace drawn
    in that cohort's color and shape throughout -- so Plotly's legend swatch for e.g.
    "Disease – TenK10K" already shows the correct color/shape pairing with no need for a
    separate, dummy shape-only legend entry."""
    fig = go.Figure()

    disease_sub = sub[sub["dataset"] == "disease"]
    for c in cohorts_shown:
        csub = disease_sub[disease_sub["cohort"] == c]
        if len(csub) == 0:
            continue
        color = COHORT_COLORS[c]
        fig.add_trace(go.Scatter(
            x=csub["_x"], y=csub["_y"], mode="markers", name=f"Disease – {c}",
            marker=dict(color=color, size=DATASET_SIZE["disease"], symbol=COHORT_SYMBOLS[c],
                        opacity=DATASET_OPACITY["disease"], line=dict(color="white", width=1)),
            customdata=cross_customdata(csub, color),
            hovertemplate="<b>%{customdata[0]}</b><extra></extra>",
        ))

    bsub_all = sub[sub["dataset"] == "matched_background"]
    for c in cohorts_shown:
        bsub = bsub_all[bsub_all["cohort"] == c]
        if len(bsub) == 0:
            continue
        color = COHORT_COLORS[c]
        fig.add_trace(go.Scatter(
            x=bsub["_x"], y=bsub["_y"], mode="markers", name=f"Background – {c}",
            marker=dict(color="grey", size=DATASET_SIZE["matched_background"], symbol=COHORT_SYMBOLS[c],
                        opacity=DATASET_OPACITY["matched_background"], line=dict(color="white", width=1)),
            customdata=cross_customdata(bsub, color),
            hovertemplate="<b>%{customdata[0]}</b><extra></extra>",
        ))

    fig.update_layout(
        template="plotly_white",
        height=600,
        dragmode="zoom",
        title="Population Spread x Fold Change of — cohorts compared, Max observed",
        xaxis_title="Population SD of repeat copy number (repeat units)",
        yaxis=dict(title="Fold-change (max observed ÷ reference)", type="log", dtick=1),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(t=100, r=40, l=70, b=60),
    )
    return fig


def build_cross_sub(datasets_shown, cohorts_shown):
    frames = [compute_cohort_rows(datasets_shown, c) for c in cohorts_shown]
    if not frames:
        empty = base_df.iloc[0:0].copy()
        empty["cohort"] = []
        return empty.assign(_x=empty.get("pop_sd"), _y=empty.get("fold_change"))
    return pd.concat(frames, ignore_index=True)


def update_cross(*_):
    cohorts_shown = [c for c in COHORT_KEYS if cohort_check_w[c].value]
    datasets_shown = ["disease"]
    if show_background_cross_w.value:
        datasets_shown.append("matched_background")

    cross_plot_out.clear_output(wait=True)
    if not cohorts_shown:
        with cross_plot_out:
            display(HTML('<p style="color:#8a8a86;">Check at least one population source above.</p>'))
        return

    sub = build_cross_sub(datasets_shown, cohorts_shown)
    fig = make_cross_figure(sub, cohorts_shown)
    with cross_plot_out:
        render_figure(fig, CROSS_PLOT_DIV_ID, CROSS_DETAIL_BOX_ID)


for w in list(cohort_check_w.values()) + [show_background_cross_w]:
    w.observe(update_cross, names="value")

update_cross()

cross_controls = widgets.VBox([
    widgets.HBox(list(cohort_check_w.values())),
    show_background_cross_w,
])

display(cross_controls, cross_plot_out)
display(detail_box_html(CROSS_DETAIL_BOX_ID))


## 5. Export the current view

Re-run this cell any time to write the table currently selected by the **main visualizer**
(section 3) widgets to `current_view.csv`.


In [ ]:
def current_view():
    cohort = cohort_source_w.value
    datasets_shown = ["disease"]
    if show_background_w.value:
        datasets_shown.append("matched_background")

    sub = compute_cohort_rows(datasets_shown, cohort)
    sub = sub.assign(population_source=cohort)
    return sub.drop(columns=["cohort_stats"]).sort_values(["dataset", "gene"])


view = current_view()
view.to_csv("current_view.csv", index=False)
print(f"Wrote current_view.csv ({len(view)} rows).")
view


## Caveats

- **Population source is single-cohort in the main view, by design — no pooling.** Combining
  population SD (or especially the maximum observed repeat length) across whichever cohorts
  happen to cover a locus would give loci covered by more, or larger, cohorts more chances to
  look extreme, independent of biology. Picking one cohort at a time (main view) or plotting
  each checked cohort's own value side by side (cross-population view) both avoid that;
  neither combines cohorts into a single pooled statistic. T2T assemblies and Illumina174k are
  both excluded from the source list for this notebook.
- **"Max observed" is a real, additional distance metric, not a stand-in for the pathogenic
  threshold.** These cohorts are general, unselected-for-disease population samples; a carrier
  of a rare disease's full pathogenic expansion essentially never turns up in a group of a few
  hundred to ~1,000 people. So the maximum observed repeat length at a disease locus will
  typically be far below its literature `pathogenic_min` — that's expected, not a data problem.
  Click a disease point to see both values side by side in the detail box; max-observed is also
  the only basis background loci can be plotted on at all, since they have no threshold by
  definition.
- **Raw maximum is outlier-sensitive.** The DAB1/HPRC256 example in section 3's markdown (max
  of 1,003 vs. a 99th percentile of 226) is driven by a single outlier genotype. The table
  computes and keeps a 99th-percentile value (`p99`) for every cohort alongside `max`
  specifically so the comparison basis could be swapped to something less outlier-prone later
  without re-deriving anything from the underlying catalogs — this notebook currently always
  uses the raw maximum.
- **The background sample is structurally matched, not population-matched.** It excludes
  known-disease-tagged loci, is restricted to gene-body regions, and restricted to the disease
  catalog's observed motif lengths (3, 4, 5, 6, 10, 12, 20, 24 bp) — but it wasn't selected for
  polymorphism the way disease loci implicitly are (a locus becomes "known disease associated"
  partly because someone noticed it varies).
- **AoU1027 completeness was fixed where the data exists, not backfilled.** The original
  63-locus catalog export only carried `AoU1027_Stdev` for this cohort; this notebook instead
  pulls the full set of statistics (max, 99th percentile, n) from the genome-wide catalog for
  the same loci. But a handful of the classic hard-to-genotype, GC-rich polyalanine-repeat loci
  (FOXL2, HOXA13, ARX, ZIC2, SOX3, TBX1, PHOX2B, RUNX2, LRP12) genuinely have **no** AoU1027
  data anywhere in the catalog, not just in the original export — those loci correctly
  disappear from both plots whenever AoU1027 is selected or checked.
- **Evidence tier floor is hardcoded, not a selector.** Only STRchive evidence tier Definitive,
  Strong, or Moderate loci are included — Limited and Disputed loci are dropped at the
  table-building step, not hidden by a UI filter. Of the original 63-locus catalog, 52 pass
  this floor (42 Definitive, 7 Moderate, 3 Strong); VWA1 is always excluded separately (see the
  comment on VWA1_GENE in section 3's code), leaving 51 disease loci on either plot.
- **Click, not hover, fills the detail box**, via a small bit of injected JavaScript rather
  than a `FigureWidget` Python callback (the comment on render_figure in section 2 explains
  why). Each plot loads plotly.js from `cdn.plot.ly` the first time it draws (needs internet
  once; cached by the browser after that) and draws into its own fixed `<div>`, so the two
  plots' click handlers and detail boxes are independent of each other. This should work the
  same way in classic Jupyter Notebook, JupyterLab, or VS Code's notebook UI; if a box doesn't
  update when you click in your setup, classic Notebook/Lab are the most tested targets.
- The export cell (section 5) only exports the **main visualizer's** current selection, not
  the cross-population comparison view.
- C9orf72's STRchive `pathogenic_min` (31 repeats) is the low end of a widely used gray zone
  (24–30 repeats); most clinical labs only call clear pathogenicity at ≥60–100+ repeats.
